# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_lib = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_lib;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)

# Đọc data

In [3]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY ID
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

## Đọc bảng An_pham_cho_muon

In [ ]:
query_Anphamchomuon = """
SELECT ID,
       Tai_lieu_ID,
       Ma_xep_gia, 
       So_the_ID,
       Ngay_muon,
       Ngay_tra,
       So_luot_gia_han,
       Note
  FROM An_pham_cho_muon
  WHERE YEAR(Ngay_muon) NOT IN (2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 
                                 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 
                                 2021, 2022, 2023, 2024)
"""
df_apcm = fetch_data_in_batches(query_Anphamchomuon, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_apcm)

Empty DataFrame
Columns: []
Index: []


C:\Users\admin\AppData\Local\Temp\ipykernel_13072\3076783788.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


## Đọc bảng Lich_su_muon_sach

In [ ]:
query_Lichsumuonsach = """
SELECT ID,
       Tai_lieu_ID,
       Ma_xep_gia, 
       So_the_ID,
       Ngay_muon,
       Ngay_tra,
       So_ngay_qua_han,
       Tien_phat
  FROM Lich_su_muon_sach
  WHERE YEAR(Ngay_muon) NOT IN (2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 
                                 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 
                                 2021, 2022, 2023, 2024)
"""
df_lscm = fetch_data_in_batches(query_Lichsumuonsach, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_lscm)

C:\Users\admin\AppData\Local\Temp\ipykernel_13072\3076783788.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


Empty DataFrame
Columns: []
Index: []


# Xử lý data

## Xử lý cột còn thiếu cho 2 bảng

In [7]:
# thêm 2 cột còn thiếu vào 
df_apcm['So_ngay_qua_han'] = None
df_apcm['Tien_phat'] = None
# chỉnh sửa cho ngày trả là None hết vì chưa trả sách
df_apcm['Ngay_tra'] = None

df_lscm['So_luot_gia_han'] = None
df_lscm['Note'] = None

print(df_apcm)
print(df_lscm)

Empty DataFrame
Columns: [So_ngay_qua_han, Tien_phat, Ngay_tra]
Index: []
Empty DataFrame
Columns: [So_luot_gia_han, Note]
Index: []


## Gộp 2 bảng lại

In [9]:
df_phieumuon = pd.concat([df_lscm, df_apcm], ignore_index=True)
df_phieumuon.rename(columns={'Tai_lieu_ID': 'ID_tai_lieu'}, inplace=True)
print(df_phieumuon)

Empty DataFrame
Columns: [So_luot_gia_han, Note, So_ngay_qua_han, Tien_phat, Ngay_tra]
Index: []


## Xử lý NaN và ""

In [10]:
df_phieumuon = df_phieumuon.replace('', None)
df_phieumuon = df_phieumuon.replace(np.nan, None)
print(df_phieumuon)

Empty DataFrame
Columns: [So_luot_gia_han, Note, So_ngay_qua_han, Tien_phat, Ngay_tra]
Index: []


## Xử lý kiểu Date

In [11]:
query_date = "SELECT Date_key FROM DIM_Date"
df_date = pd.read_sql(query_date, conn_dwh_lib)
date_ids = set(df_date['Date_key'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_phieumuon['Ngay_muon'] = pd.to_datetime(df_phieumuon['Ngay_muon'], errors='coerce')
df_phieumuon['Ngay_tra'] = pd.to_datetime(df_phieumuon['Ngay_tra'], errors='coerce')
df_phieumuon['Ngay_muon'] = df_phieumuon['Ngay_muon'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_phieumuon['Ngay_tra'] = df_phieumuon['Ngay_tra'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)

print(df_phieumuon[['Ngay_muon', 'Ngay_tra']])

C:\Users\admin\AppData\Local\Temp\ipykernel_13072\3472876002.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_lib)


KeyError: 'Ngay_muon'

## Xử lý ID_tai_lieu

In [169]:
query_Tailieu = "SELECT ID_tai_lieu FROM DIM_Tai_lieu"
df_tailieu = pd.read_sql(query_Tailieu, conn_dwh_lib)
tailieu_ids = set(df_tailieu['ID_tai_lieu'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_tai_lieu của bảng DIM_Tai_lieu hay không ?
df_phieumuon['ID_tai_lieu'] = df_phieumuon['ID_tai_lieu'].apply(lambda x: x if pd.notna(x) and x in tailieu_ids else 0)
print(df_phieumuon[['ID_tai_lieu']])

   ID_tai_lieu
0       7554.0
1       7553.0
2       7553.0
3      11089.0
4      11096.0
5      22194.0
6      14668.0
7      12495.0
8      12368.0
9      11090.0


C:\Users\admin\AppData\Local\Temp\ipykernel_18776\1236753618.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tailieu = pd.read_sql(query_Tailieu, conn_dwh_lib)


## Xử lý ID_xep_gia

In [170]:
query_Xepgia = "SELECT ID_xep_gia, ID_tai_lieu, Ma_xep_gia FROM DIM_Xep_gia"
df_xepgia = pd.read_sql(query_Xepgia, conn_dwh_lib)
# Gán ID_xep_gia từ df_xep_gia vào df_phieu_muon_sach
df_phieumuon['ID_xep_gia'] = None
df_phieumuon['ID_xep_gia'] = df_phieumuon.apply(lambda row: df_xepgia.loc[
                                                (df_xepgia['ID_tai_lieu'] == row['ID_tai_lieu']) & 
                                                (df_xepgia['Ma_xep_gia'] == row['Ma_xep_gia']), 
                                                'ID_xep_gia'
                                                ].iloc[0] if not df_xepgia[
                                                    (df_xepgia['ID_tai_lieu'] == row['ID_tai_lieu']) & 
                                                    (df_xepgia['Ma_xep_gia'] == row['Ma_xep_gia'])
                                                    ].empty else 0,
                                                    axis=1)
print(df_phieumuon[['ID_tai_lieu', 'Ma_xep_gia', 'ID_xep_gia']])

C:\Users\admin\AppData\Local\Temp\ipykernel_18776\1872470284.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_xepgia = pd.read_sql(query_Xepgia, conn_dwh_lib)


   ID_tai_lieu Ma_xep_gia  ID_xep_gia
0       7554.0   ns000002           0
1       7553.0   ns000001      218317
2       7553.0   ns000001      218317
3      11089.0  GT0005338      241160
4      11096.0  GT0066836      315718
5      22194.0  GT0165834      485966
6      14668.0  GT0066078      314619
7      12495.0  GT0039796      277035
8      12368.0  GT0034271      270935
9      11090.0  GT0004584      244152


## Xử lý ID_ban_doc

### Đọc từ libol để đổi ID sang So_the

In [171]:
query_Ban_doc = "SELECT ID, dbo.DecodeUTF8String(So_the) AS So_the FROM Ban_doc"
df_bandoc = pd.read_sql(query_Ban_doc, conn_libol)

df_phieumuon['ID_ban_doc'] = None
df_phieumuon['ID_ban_doc'] = df_phieumuon['So_the_ID'].apply(
                                                            lambda x: df_bandoc.loc[df_bandoc['ID'] == x, 
                                                                                    'So_the'].iloc[0] 
                                                            if not df_bandoc[df_bandoc['ID'] == x].empty else 0)
print(df_phieumuon[['So_the_ID', 'ID_ban_doc']])

C:\Users\admin\AppData\Local\Temp\ipykernel_18776\2702041677.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bandoc = pd.read_sql(query_Ban_doc, conn_libol)


  So_the_ID ID_ban_doc
0    6346.0  N97105632
1    6346.0  N97105632
2    6346.0  N97105632
3      None          0
4      None          0
5      None          0
6      None          0
7      None          0
8      None          0
9      None          0


### Kiểm tra lại so db dwh_lib

In [172]:
query_Bandoc = "SELECT ID_ban_doc FROM DIM_Ban_doc"
df_bandoc = pd.read_sql(query_Bandoc, conn_dwh_lib)
bandoc_ids = set(df_bandoc['ID_ban_doc'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_tai_lieu của bảng DIM_Tai_lieu hay không ?
df_phieumuon['ID_ban_doc'] = df_phieumuon['ID_ban_doc'].apply(lambda x: x if pd.notna(x) and x in bandoc_ids else 0)
print(df_phieumuon[['So_the_ID', 'ID_ban_doc']])

  So_the_ID ID_ban_doc
0    6346.0  N97105632
1    6346.0  N97105632
2    6346.0  N97105632
3      None          0
4      None          0
5      None          0
6      None          0
7      None          0
8      None          0
9      None          0


C:\Users\admin\AppData\Local\Temp\ipykernel_18776\3154858834.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bandoc = pd.read_sql(query_Bandoc, conn_dwh_lib)


# Load data

## [Nếu cần] Clear bảng

In [173]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM FACT_Phieu_muon_sach"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Load data vào bảng DIM_Xep_gia

In [174]:
cursor_dwh = conn_dwh_lib.cursor()

# Lệnh INSERT cho từng hàng trong df_phieumuon
insert_query = """
INSERT INTO FACT_Phieu_muon_sach (
    ID_phieu_muon, 
    ID_tai_lieu, ID_xep_gia,
    ID_ban_doc,
    Ngay_muon, Ngay_tra,
    So_luot_gia_han, So_ngay_qua_han,
    Tien_phat, Ghi_chu
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""

data_to_insert = [
    (
        row['ID'], 
        row['ID_tai_lieu'], row['ID_xep_gia'], 
        row['ID_ban_doc'], 
        row['Ngay_muon'], row['Ngay_tra'],
        row['So_luot_gia_han'], row['So_ngay_qua_han'],
        row['Tien_phat'], row['Note']
    )
    for index, row in df_phieumuon.iterrows()
]
# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)
# Commit thay đổi
conn_dwh_lib.commit()
# Đóng cursor và kết nối
cursor_dwh.close()
conn_dwh_lib.close()